In [ ]:
!pip install google-play-scraper -q
# Install & import required library
import pandas as pd
import time
import os
from datetime import datetime
from google_play_scraper import app, Sort, reviews


print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [ ]:
PROJECT_DIR = "/kaggle/working"
CONFIG = {
    # App identifier for Shopee Indonesia
    'app_id': 'com.shopee.id',
    
    # Scraping parameters
    'lang': 'id',  # Indonesian reviews
    'country': 'id',  # Indonesia region
    'sort': Sort.NEWEST,  # Get newest reviews first
    
    # Phase 1: Exploratory scraping (no keyword filter)
    # Phase 2: Set keywords for targeted scraping
    'use_keyword_filter': True,  # Set True for Phase 2
    'seed_keywords': {
        "Aplikasi":  ["aplikasi", "app",  "bug", "error", "keluar", "lag", 
                      "loading", "fitur", "menu", "tombol", "akun", 
                      "blokir", "daftar", "login", "masuk", "password", "verifikasi", "cari"],
        "Pengiriman": ["pengiriman", "kirim","antar", "kurir", "ongkir", "paket", "resi",
                       "tracking", "lacak"],
        "Produk": ["asli", "bahan", "kualitas", "mutu", "original", "palsu", "produk",
                   "rusak", "deskripsi produk", "barang"],
        "Harga": ["harga", "diskon", "kupon", "mahal", "murah", "promo", "voucher"],
        "Pembayaran": ["bayar", "cod", "kartu", "refund", "saldo", "transfer"],
        "Layanan Pelanggan (CS)": ["balas","customer service", "jawab", "komplain", "kontak", "respon",
                                   "solusi", "ramah"],
        "Penjual": ["seller","toko"]
    },
    
    # Scraping limits
    'batch_size': 200,  # Reviews per batch request
    'max_reviews_phase1': 1000,  # For exploratory analysis
    'min_threshold_per_aspect': 100,  # For Phase 2 (reviews per aspect)
    
    # Rate limiting (be nice to Google's servers!)
    'delay_between_batches': 3,  # seconds
    'delay_on_error': 10,  # seconds
    
    # Output settings
    'output_dir': PROJECT_DIR,  # Kaggle default working directory
    'checkpoint_file': 'scraping_checkpoint.csv',
}

print("✅ Configuration loaded!")
print(f"📱 Target app: Shopee Indonesia ({CONFIG['app_id']})")
print(f"🎯 Phase: {'1 - Exploratory' if not CONFIG['use_keyword_filter'] else '2 - Targeted'}")

✅ Configuration loaded!
📱 Target app: Shopee Indonesia (com.shopee.id)
🎯 Phase: 2 - Targeted


In [ ]:
# Input: phase (str) — nama fase/label untuk file, default "phase1"
# Proses: membuat string timestamp menggunakan datetime.now() dan menyimpannya ke variabel `timestamp`.
#         *Catatan:* variabel `timestamp` dibuat tetapi **tidak digunakan** pada baris return saat ini.
# Output: mengembalikan nama file (str) dalam format "shopee_reviews_{phase}.csv"
# Kegunaan: digunakan untuk menghasilkan nama file output yang konsisten berdasarkan fase proses (mis. "phase1").
#           Jika Anda ingin menambahkan timestamp ke nama file, ganti return agar memasukkan `timestamp`.
def create_output_filename(phase="phase1"):
    """Generate timestamped filename"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"shopee_reviews_{phase}.csv"
# Input: text (str) — teks review yang akan diperiksa; aspect_keywords (dict) — mapping dari aspek -> list kata kunci
# Proses: 
#   1. Periksa apakah text kosong/None; jika kosong kembalikan list kosong.
#   2. Normalisasi teks ke huruf kecil (lowercase) untuk pencarian case-insensitive.
#   3. Iterasi tiap aspek di aspect_keywords; untuk setiap aspek, cek apakah ada keyword (juga lowercase)
#      yang muncul sebagai substring di dalam text_lower. Jika ada, tambahkan nama aspek ke matched_aspects.
#   4. Metode ini menggunakan pencocokan substring sederhana (bukan regex atau tokenisasi),
#      sehingga dapat menghasilkan false positives untuk kata yang merupakan bagian dari kata lain.
# Output: list of matched aspek (list of str) — nama-nama aspek yang ditemukan dalam teks; bisa kosong.
# Kegunaan: mengklasifikasikan atau menandai review berdasarkan kehadiran kata kunci aspek (mis. "harga", "kualitas").
def contains_keywords(text, aspect_keywords):
    """Check if text contains any keyword from aspect_keywords dict"""
    if not text:
        return []
    
    text_lower = text.lower()
    matched_aspects = []
    
    for aspect, keywords in aspect_keywords.items():
        if any(keyword.lower() in text_lower for keyword in keywords):
            matched_aspects.append(aspect)
    
    return matched_aspects
# Input: review (dict-like) — objek/dict review yang berisi berbagai field (reviewId, userName, content, score, dll.)
# Proses:
#   1. Mengambil nilai untuk field-field penting dari objek review menggunakan review.get(key) sehingga
#      tidak melempar KeyError bila kunci tidak ada.
#   2. Mengumpulkan field-field tersebut ke dalam dict baru dengan key yang konsisten.
# Output: dict yang hanya berisi field relevan:
#         {'reviewId','userName','userImage','content','score',
#          'reviewCreatedVersion','at','replyContent','repliedAt'}
# Kegunaan: menormalisasi/menyaring objek review mentah menjadi struktur data yang lebih kecil dan konsisten
#           untuk disimpan, diproses lebih lanjut (analisis, clustering, dsb.), atau diekspor ke CSV/DB.
def extract_review_data(review):
    """Extract relevant fields from review object"""
    return {
        'reviewId': review.get('reviewId'),
        'userName': review.get('userName'),
        'userImage': review.get('userImage'),
        'content': review.get('content'),
        'score': review.get('score'),
        'reviewCreatedVersion': review.get('reviewCreatedVersion'),
        'at': review.get('at'),
        'replyContent': review.get('replyContent'),
        'repliedAt': review.get('repliedAt'),
    }
# Input: data (list of dict) — daftar review/record yang ingin disimpan sebagai checkpoint; filepath (str) — path file CSV tujuan
# Proses:
#   1. Konversi `data` menjadi pandas.DataFrame.
#   2. Menyimpan DataFrame ke CSV menggunakan encoding 'utf-8-sig' dan tanpa index.
#   3. Mencetak pesan informasi jumlah review yang disimpan.
# Output: Tidak mengembalikan nilai (None). Sisi efek: file CSV dibuat/diperbarui pada filepath.
# Kegunaan: menyimpan progress scraping / pemrosesan (checkpoint) sehingga pekerjaan bisa dilanjutkan dari titik ini
#           jika proses terhenti atau untuk pengecekan manual.
def save_checkpoint(data, filepath):
    """Save progress checkpoint"""
    df = pd.DataFrame(data)
    df.to_csv(filepath, index=False, encoding='utf-8-sig')
    print(f"💾 Checkpoint saved: {len(data)} reviews")
# Input: filepath (str) — path file CSV checkpoint yang mungkin sudah ada
# Proses:
#   1. Mengecek apakah file ada menggunakan os.path.exists.
#   2. Jika ada, membaca CSV ke pandas.DataFrame, mencetak pesan berapa banyak baris yang dimuat,
#      lalu mengembalikan data sebagai list of dict (df.to_dict('records')).
#   3. Jika file tidak ada, mengembalikan list kosong.
# Output: list of dict — data yang dimuat dari CSV, atau [] jika tidak ada file.
# Kegunaan: memulihkan state/hasil sementara sebelumnya (checkpoint) sehingga pipeline dapat dilanjutkan tanpa
#           mengulang pekerjaan yang sudah selesai.
def load_checkpoint(filepath):
    """Load existing checkpoint if available"""
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"📂 Loaded checkpoint: {len(df)} reviews")
        return df.to_dict('records')
    return []

print("✅ Helper functions loaded!")

✅ Helper functions loaded!


In [4]:
# Input: config (dict) — konfigurasi untuk scraping. Harus berisi minimal kunci:
#        - 'max_reviews_phase1' (int): target jumlah review untuk Phase 1
#        - 'output_dir' (str): direktori output untuk menyimpan file CSV dan checkpoint
#        - 'checkpoint_file' (str): nama file checkpoint (CSV)
#        - 'app_id' (str/int): identifier aplikasi untuk pemanggilan API reviews(...)
#        - 'lang' (str): kode bahasa untuk reviews(...)
#        - 'country' (str): kode negara untuk reviews(...)
#        - 'sort' (str): urutan pengambilan reviews (mis. 'newest')
#        - 'batch_size' (int): jumlah review yang diminta tiap batch
#        - 'delay_between_batches' (float/int): jeda (detik) antar batch untuk rate limiting
# Proses / Cara kerja (detail langkah demi langkah):
#   1. Menampilkan header informatif ke console (bagian "PHASE 1: EXPLORATORY SCRAPING").
#   2. Menyiapkan struktur data awal:
#        - all_reviews: list kosong (atau hasil load_checkpoint jika ada file checkpoint sebelumnya)
#        - continuation_token: None untuk memulai paging API
#   3. Menentukan path checkpoint dengan os.path.join(config['output_dir'], config['checkpoint_file'])
#      lalu memanggil load_checkpoint(checkpoint_path) untuk memulihkan progress bila ada.
#   4. Memasuki loop utama: while len(all_reviews) < config['max_reviews_phase1']:
#        a. Mencetak informasi jumlah review saat ini.
#        b. Memanggil fungsi reviews(...) dengan parameter dari config untuk meminta batch review.
#           Fungsi reviews diasumsikan mengembalikan tuple (result, continuation_token).
#        c. Jika result kosong/None -> hentikan loop (tidak ada lagi review).
#        d. Untuk setiap review di result:
#             - memanggil extract_review_data(review) untuk menormalisasi/menyaring field penting
#             - menambahkan dict hasil ke all_reviews
#        e. Mencetak ringkasan batch selesai dan menyimpan checkpoint menggunakan save_checkpoint(all_reviews, checkpoint_path).
#        f. Memeriksa apakah target jumlah review telah tercapai; jika ya, keluar dari loop.
#        g. Jika belum tercapai, menunggu selama config['delay_between_batches'] detik untuk menghormati rate limit.
#   5. Menangani pengecualian umum (Exception) pada blok try/except:
#        - Mencetak pesan error, menyimpan progress saat ini ke checkpoint, lalu melanjutkan ke langkah simpan akhir.
#   6. Setelah loop selesai (normal atau akibat exception), menyimpan hasil akhir:
#        - Membuat nama file output dengan create_output_filename("phase1") dan menyimpan CSV ke output_dir
#        - Menyusun DataFrame pandas dari all_reviews dan menulis ke CSV (utf-8-sig, tanpa index)
#   7. Mencetak ringkasan hasil akhir, langkah selanjutnya, dan mengembalikan DataFrame.
# Output:
#   - Menghasilkan side effects: file checkpoint diperbarui setiap batch; file CSV output akhir disimpan di disk.
#   - Return value: pandas.DataFrame (df) yang berisi semua review yang dikumpulkan (bisa kurang dari target jika stok habis).
# Kegunaan:
#   - Fungsi ini melakukan _exploratory scraping_ untuk mengambil sampel review acak/bertahap sehingga tim bisa
#     menganalisis isi review secara manual dan menyusun taxonomy/aspect seed keywords untuk fase selanjutnya.
#   - Cocok dipakai sebagai fase awal pipeline data collection sebelum tahap labeling/aspect-discovery.
# Catatan penting / Dependensi:
#   - Mengandalkan fungsi/variabel eksternal: load_checkpoint, save_checkpoint, reviews, extract_review_data,
#     create_output_filename, serta modul: os, time, pandas as pd.
#   - Pastikan direktori output ada dan config berisi semua kunci yang diperlukan agar fungsi berjalan lancar.
def scrape_phase1_exploratory(config):
    """
    Phase 1: Exploratory scraping
    Goal: Get random sample to identify aspects and create taxonomy
    """
    print("="*70)
    print("🔍 PHASE 1: EXPLORATORY SCRAPING")
    print("="*70)
    print(f"Target: {config['max_reviews_phase1']} reviews")
    print(f"Purpose: Identify aspects for taxonomy creation")
    print()
    
    all_reviews = []
    continuation_token = None
    
    checkpoint_path = os.path.join(config['output_dir'], config['checkpoint_file'])
    all_reviews = load_checkpoint(checkpoint_path)
    
    try:
        while len(all_reviews) < config['max_reviews_phase1']:
            print(f"📥 Fetching batch... (Current: {len(all_reviews)} reviews)")
            
            result, continuation_token = reviews(
                config['app_id'],
                lang=config['lang'],
                country=config['country'],
                sort=config['sort'],
                count=config['batch_size'],
                continuation_token=continuation_token
            )
            
            if not result:
                print("⚠️ No more reviews available")
                break
            
            # Extract and store review data
            for review in result:
                review_data = extract_review_data(review)
                all_reviews.append(review_data)
            
            print(f"✅ Batch completed: +{len(result)} reviews")
            
            # Save checkpoint every batch
            save_checkpoint(all_reviews, checkpoint_path)
            
            # Check if we reached target
            if len(all_reviews) >= config['max_reviews_phase1']:
                print(f"🎯 Target reached: {len(all_reviews)} reviews")
                break
            
            # Rate limiting
            print(f"⏳ Waiting {config['delay_between_batches']}s before next batch...")
            time.sleep(config['delay_between_batches'])
            
    except Exception as e:
        print(f"❌ Error occurred: {str(e)}")
        print(f"💾 Saving progress... ({len(all_reviews)} reviews)")
        save_checkpoint(all_reviews, checkpoint_path)
    
    # Save final results
    output_file = os.path.join(config['output_dir'], create_output_filename("phase1"))
    df = pd.DataFrame(all_reviews)
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print()
    print("="*70)
    print("✅ PHASE 1 COMPLETED")
    print("="*70)
    print(f"Total reviews scraped: {len(all_reviews)}")
    print(f"Output file: {output_file}")
    print()
    print("📊 Next steps:")
    print("1. Manual analysis of reviews to identify aspects")
    print("2. Create aspect taxonomy and seed keywords")
    print("3. Update CONFIG['seed_keywords'] for Phase 2")
    
    return df

print("✅ Phase 1 function loaded!")

✅ Phase 1 function loaded!


In [1]:
# Input:
#   - config (dict): konfigurasi untuk Phase 2. Harus berisi minimal:
#       * 'seed_keywords' (dict): mapping aspek -> list kata kunci (harus diisi sebelum menjalankan Phase 2)
#       * 'min_threshold_per_aspect' (int): minimal jumlah review yang ingin dikumpulkan per aspek
#       * 'output_dir' (str): direktori output untuk menyimpan hasil dan checkpoint
#       * 'app_id', 'lang', 'country', 'sort', 'batch_size' (untuk pemanggilan google_play_scraper.reviews)
#       * 'delay_between_batches' (float/int): jeda (detik) antar batch untuk rate limiting
# Proses / Cara kerja (detail langkah demi langkah):
#   1. Tampilkan header informasi fase dan periksa keberadaan config['seed_keywords']; jika tidak ada, hentikan.
#   2. Import gp_reviews dari google_play_scraper secara eksplisit untuk menghindari ambiguitas nama fungsi `reviews`.
#   3. Inisialisasi struktur:
#       - aspect_review_tracker: dict tiap aspek -> list review (tracking per-aspek)
#       - collected_reviews: list semua review relevan yang terkumpul
#       - seen_ids: set untuk menghindari duplikat review (berdasarkan reviewId)
#       - continuation_token: None (dipakai untuk paging API)
#       - aspect_names: list nama aspek untuk pembuatan kolom indikator nanti
#       - checkpoint_path: path file checkpoint 'phase2_checkpoint.csv' di output_dir
#   4. Bila checkpoint ada, coba baca CSV dan rekontruksi state:
#       - tambahkan setiap baris ke collected_reviews dan seen_ids
#       - jalankan contains_keywords pada kolom 'content' untuk menempatkan review ke aspect_review_tracker
#       - jika pemuatan checkpoint gagal, tampilkan warning dan mulai fresh
#   5. Masuk loop utama dengan safety limit max_batches (default 50) untuk mencegah infinite loop:
#       a. Hitung aspek yang telah mencapai threshold dan aspek yang tersisa.
#       b. Jika tidak ada aspek tersisa -> selesai.
#       c. Cetak progress per aspek.
#       d. Panggil gp_reviews(...) untuk mengambil batch; tangani errors eksplisit:
#           - TypeError: beri saran instal versi tertentu dari library
#           - Exception lainnya: hentikan scraping untuk mencegah korupsi data
#       e. Jika fetch_result kosong -> hentikan (tidak ada review lagi).
#       f. Iterasi tiap review di fetch_result:
#           - lewati duplikat berdasarkan reviewId
#           - cek kecocokan aspek dengan contains_keywords
#           - jika cocok: normalisasi review via extract_review_data, tambahkan field 'matched_aspects',
#             tambahkan ke collected_reviews, seen_ids, dan ke aspect_review_tracker (jika aspek belum capai threshold)
#       g. Simpan checkpoint (save_checkpoint) setelah tiap batch jika ada collected_reviews
#       h. Tambah counter batch_num dan tunggu delay_between_batches detik
#   6. Tangani interupsi (KeyboardInterrupt) dan exception umum pada loop: simpan progress ke checkpoint sebelum keluar.
#   7. Setelah loop, jika ada collected_reviews:
#       - Bangun DataFrame dari collected_reviews
#       - Pastikan kolom 'matched_aspects' ada dan normalisasi isinya ke string (hindari NaN)
#       - Buat kolom indikator (0/1) untuk tiap aspek berdasarkan apakah aspek ada di 'matched_aspects'
#       - Susun ulang kolom sehingga kolom aspek berada di bagian akhir, lalu simpan CSV hasil
#       - Cetak ringkasan dan kembalikan DataFrame
#      Jika tidak ada collected_reviews:
#       - Cetak pesan informatif, saran perbaikan, dan kembalikan DataFrame kosong
# Output:
#   - Side effects: file checkpoint periodik ('phase2_checkpoint.csv') dan file output final 'shopee_reviews_phase2.csv' (jika data ada)
#   - Return value: pandas.DataFrame berisi semua review yang terkumpul (dengan kolom indikator per aspek);
#                   jika tidak ada data, kembalikan DataFrame kosong
# Kegunaan:
#   - Melakukan scraping terarah dari Google Play Store untuk mengumpulkan distribusi aspek seimbang berdasarkan seed keywords.
#   - Menghemat effort anotasi dengan hanya menyimpan review yang relevan untuk tiap aspek dan menandai tiap review
#     apakah berkaitan dengan aspek tertentu (kolom indikator).
# Catatan / Dependensi:
#   - Mengandalkan fungsi/variabel eksternal: gp_reviews (google_play_scraper.reviews), contains_keywords,
#     extract_review_data, save_checkpoint, create_output_filename, serta modul: os, time, pandas as pd.
#   - Checkpoint path di-hardcode 'phase2_checkpoint.csv' (relatif ke output_dir); bisa digantikan dengan config key untuk konsistensi.
#   - Safety: max_batches membatasi iterasi agar proses tidak berjalan tanpa henti jika aspek sulit ditemukan.
#   - Pastikan config['app_id'] benar dan seed_keywords tidak terlalu sempit, serta siap menghadapi rate limiting dari Google Play.
def scrape_phase2_targeted(config):
    """
    Phase 2: Targeted scraping with keyword filtering
    Goal: Get balanced distribution of aspects based on seed keywords
    FIXED VERSION: Better error handling and variable naming
    """
    print("="*70)
    print("🎯 PHASE 2: TARGETED SCRAPING")
    print("="*70)
    
    if not config['seed_keywords']:
        print("❌ ERROR: Please define seed_keywords in CONFIG first!")
        print("Run Phase 1 exploratory analysis to identify aspects.")
        return None
    
    print(f"Aspects to collect: {list(config['seed_keywords'].keys())}")
    print(f"Minimum threshold per aspect: {config['min_threshold_per_aspect']} reviews")
    print()
    
    # Import reviews function explicitly to avoid confusion
    from google_play_scraper import reviews as gp_reviews
    
    # Track reviews per aspect (renamed to avoid confusion)
    aspect_review_tracker = {aspect: [] for aspect in config['seed_keywords'].keys()}
    collected_reviews = []
    seen_ids = set()
    continuation_token = None
    
    # Get aspect names untuk kolom nanti
    aspect_names = list(config['seed_keywords'].keys())
    
    checkpoint_path = os.path.join(config['output_dir'], 'phase2_checkpoint.csv')
    
    # Check if checkpoint exists
    if os.path.exists(checkpoint_path):
        try:
            checkpoint_df = pd.read_csv(checkpoint_path)
            print(f"📂 Loaded checkpoint: {len(checkpoint_df)} reviews")
            
            # Reconstruct tracking from checkpoint
            for _, row in checkpoint_df.iterrows():
                review_id = row['reviewId']
                seen_ids.add(review_id)
                collected_reviews.append(row.to_dict())
                
                # Re-check keywords for aspect tracking
                matched = contains_keywords(row.get('content', ''), config['seed_keywords'])
                for aspect in matched:
                    if aspect in aspect_review_tracker:
                        aspect_review_tracker[aspect].append(row.to_dict())
        except Exception as e:
            print(f"⚠️ Warning: Could not load checkpoint properly: {e}")
            print("Starting fresh...")
    
    try:
        batch_num = 0
        max_batches = 50  # Safety limit to prevent infinite loop
        
        while batch_num < max_batches:
            # Check if all aspects reached threshold
            completed = [asp for asp, revs in aspect_review_tracker.items() 
                        if len(revs) >= config['min_threshold_per_aspect']]
            remaining = [asp for asp in config['seed_keywords'].keys() 
                        if asp not in completed]
            
            if not remaining:
                print("🎉 All aspects reached minimum threshold!")
                break
            
            print(f"\n📊 Aspect Progress:")
            for aspect, review_list in aspect_review_tracker.items():
                status = "✅" if len(review_list) >= config['min_threshold_per_aspect'] else "⏳"
                print(f"  {status} {aspect}: {len(review_list)}/{config['min_threshold_per_aspect']}")
            
            print(f"\n📥 Fetching batch {batch_num + 1}...")
            
            # Fetch reviews with explicit error handling
            try:
                fetch_result, continuation_token = gp_reviews(
                    config['app_id'],
                    lang=config['lang'],
                    country=config['country'],
                    sort=config['sort'],
                    count=config['batch_size'],
                    continuation_token=continuation_token
                )
            except TypeError as te:
                print(f"❌ TypeError during fetch: {te}")
                print("This is likely a google-play-scraper library issue.")
                print("Try: pip install google-play-scraper==1.2.4")
                break
            except Exception as e:
                print(f"❌ Unexpected error during fetch: {type(e).__name__}: {e}")
                print("Stopping scraping to prevent data corruption.")
                break
            
            if not fetch_result:
                print("⚠️ No more reviews available from Google Play Store")
                break
            
            matched_in_batch = 0
            for review_item in fetch_result:
                rev_id = review_item.get('reviewId')
                
                # Skip duplicates
                if rev_id in seen_ids:
                    continue
                
                review_text = review_item.get('content', '')
                matched_aspects = contains_keywords(review_text, config['seed_keywords'])
                
                # Only keep reviews that match at least one aspect
                if matched_aspects:
                    review_data = extract_review_data(review_item)
                    review_data['matched_aspects'] = ','.join(matched_aspects)
                    
                    collected_reviews.append(review_data)
                    seen_ids.add(rev_id)
                    matched_in_batch += 1
                    
                    # Add to aspect-specific tracking
                    for aspect in matched_aspects:
                        if aspect in aspect_review_tracker:
                            if len(aspect_review_tracker[aspect]) < config['min_threshold_per_aspect']:
                                aspect_review_tracker[aspect].append(review_data)
            
            print(f"✅ Batch {batch_num + 1}: {matched_in_batch}/{len(fetch_result)} reviews matched keywords")
            
            # Save checkpoint
            if collected_reviews:
                save_checkpoint(collected_reviews, checkpoint_path)
            
            batch_num += 1
            
            # Rate limiting
            time.sleep(config['delay_between_batches'])
            
    except KeyboardInterrupt:
        print("\n⚠️ Scraping interrupted by user")
        print(f"💾 Saving progress... ({len(collected_reviews)} reviews)")
        if collected_reviews:
            save_checkpoint(collected_reviews, checkpoint_path)
    except Exception as e:
        print(f"\n❌ Unexpected error in main loop: {type(e).__name__}: {e}")
        print(f"💾 Saving progress... ({len(collected_reviews)} reviews)")
        if collected_reviews:
            save_checkpoint(collected_reviews, checkpoint_path)
    
    # Save final results
    if collected_reviews:
        output_file = os.path.join(config['output_dir'], create_output_filename("phase2"))
        df = pd.DataFrame(collected_reviews)
    
        # Pastikan kolom 'matched_aspects' ada (jika checkpoint berisi review tanpa kolom ini)
        if 'matched_aspects' not in df.columns:
            df['matched_aspects'] = ''
    
        # Normalisasi jadi string (hindari NaN)
        df['matched_aspects'] = df['matched_aspects'].fillna('').astype(str)
    
        # Buat kolom indikator untuk setiap aspek (1 = ada, 0 = tidak)
        for aspect in aspect_names:
            # jika Anda ingin boolean gunakan: lambda s: aspect in ...
            df[aspect] = df['matched_aspects'].apply(
                lambda s, asp=aspect: 1 if asp in [a.strip() for a in s.split(',') if a.strip()] else 0
            )
    
        # Susun ulang kolom: kolom non-aspek dulu, lalu kolom aspek di akhir
        non_aspect_cols = [c for c in df.columns.tolist() if c not in aspect_names]
        final_cols = non_aspect_cols + aspect_names
        df = df[final_cols]
    
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
        print()
        print("="*70)
        print("✅ PHASE 2 COMPLETED")
        print("="*70)
        print(f"Total reviews collected: {len(collected_reviews)}")
        print(f"Output file: {output_file}")
        print()
        print("📊 Final Aspect Distribution:")
        for aspect in aspect_names:
            print(f"  {aspect}: {int(df[aspect].sum())} reviews")
        print()
        print("📝 Next step: Manual annotation of (aspect, sentiment) pairs")
    
        return df


print("✅ Phase 2 function loaded!")

✅ Phase 2 function loaded!


In [6]:
print("\n" + "="*70)
print("🛍️  SHOPEE GOOGLE PLAY STORE REVIEW SCRAPER")
print("="*70 + "\n")

# Run Phase 1
df_phase2 = scrape_phase2_targeted(CONFIG)

# Show summary
if df_phase2 is not None:
    print("\n📈 Dataset Summary:")
    print(f"  Total reviews: {len(df_phase2)}")
    print(f"  Average score: {df_phase2['score'].mean():.2f}/5")
    
    # Show aspect distribution
    aspect_counts = {}
    for aspects_str in df_phase2['matched_aspects']:
        for aspect in aspects_str.split(','):
            aspect_counts[aspect] = aspect_counts.get(aspect, 0) + 1
    
    print("\n📊 Aspect Distribution:")
    for aspect, count in sorted(aspect_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {aspect}: {count} reviews")

print("✅ Script ready! Run cells sequentially to start scraping.")


🛍️  SHOPEE GOOGLE PLAY STORE REVIEW SCRAPER

🎯 PHASE 2: TARGETED SCRAPING
Aspects to collect: ['Aplikasi', 'Pengiriman', 'Produk', 'Harga', 'Pembayaran', 'Layanan Pelanggan (CS)', 'Penjual']
Minimum threshold per aspect: 100 reviews


📊 Aspect Progress:
  ⏳ Aplikasi: 0/100
  ⏳ Pengiriman: 0/100
  ⏳ Produk: 0/100
  ⏳ Harga: 0/100
  ⏳ Pembayaran: 0/100
  ⏳ Layanan Pelanggan (CS): 0/100
  ⏳ Penjual: 0/100

📥 Fetching batch 1...
✅ Batch 1: 78/200 reviews matched keywords
💾 Checkpoint saved: 78 reviews

📊 Aspect Progress:
  ⏳ Aplikasi: 33/100
  ⏳ Pengiriman: 25/100
  ⏳ Produk: 5/100
  ⏳ Harga: 14/100
  ⏳ Pembayaran: 11/100
  ⏳ Layanan Pelanggan (CS): 13/100
  ⏳ Penjual: 11/100

📥 Fetching batch 2...
✅ Batch 2: 69/200 reviews matched keywords
💾 Checkpoint saved: 147 reviews

📊 Aspect Progress:
  ⏳ Aplikasi: 59/100
  ⏳ Pengiriman: 57/100
  ⏳ Produk: 17/100
  ⏳ Harga: 34/100
  ⏳ Pembayaran: 19/100
  ⏳ Layanan Pelanggan (CS): 29/100
  ⏳ Penjual: 16/100

📥 Fetching batch 3...
✅ Batch 3: 73/200 

In [ ]:
import pandas as pd
# 1️⃣ Load
df = pd.read_csv('/kaggle/working/shopee_reviews_phase2.csv')

# 2️⃣ Descriptive stats
print("=== DESCRIPTIVE STATS (NUMERIC) ===")
print(df.describe())

# 3️⃣ Descriptive untuk teks (optional)
df['text_length'] = df['content'].fillna('').str.len()
print("\n=== TEXT LENGTH STATS ===")
print(df['text_length'].describe())

# 4️⃣ Missing values
print("\n=== MISSING VALUES ===")
print(df.isna().sum())

# 5️⃣ Duplicate

print("\n=== DUPLICATES ===")
cols = list(df.columns)
subset = cols[cols.index('userName'):cols.index('matched_aspects')+1]
duplicate_count = df.duplicated(subset=subset, keep='first').sum()
print(duplicate_count)
print(f"Columns: {df.columns.tolist()}")

=== DESCRIPTIVE STATS (NUMERIC) ===
             score     Aplikasi   Pengiriman       Produk        Harga  \
count  1043.000000  1043.000000  1043.000000  1043.000000  1043.000000   
mean      3.727709     0.475551     0.337488     0.099712     0.243528   
std       1.720562     0.499641     0.473080     0.299760     0.429417   
min       1.000000     0.000000     0.000000     0.000000     0.000000   
25%       2.000000     0.000000     0.000000     0.000000     0.000000   
50%       5.000000     0.000000     0.000000     0.000000     0.000000   
75%       5.000000     1.000000     1.000000     0.000000     0.000000   
max       5.000000     1.000000     1.000000     1.000000     1.000000   

        Pembayaran  Layanan Pelanggan (CS)      Penjual  
count  1043.000000             1043.000000  1043.000000  
mean      0.099712                0.146692     0.101630  
std       0.299760                0.353969     0.302306  
min       0.000000                0.000000     0.000000  
25%    